In [1]:
import datetime
import os

import duckdb

In [2]:
# Define paths
base_path_merge = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged"
base_path_filtered = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\filtered"
output_dir = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged"

merged_file = os.path.join(base_path_merge, "merged_bookings_measurements_cleaned.parquet")
filtered_materials_file = os.path.join(base_path_filtered, "filtered_materials_encoded_time_cut.parquet")
merged_full_file = os.path.join(output_dir, "merged_full.parquet")

TIME_WINDOW_DAYS = 14

In [3]:
# Connect to DuckDB (in-memory engine, but will spill to disk if needed)
con = duckdb.connect()

# Get start_time, end_time as before
min_created_at_materials = con.execute(f"SELECT MIN(created_at) FROM '{filtered_materials_file}'").fetchone()[0]
min_created_at_merged = con.execute(f"SELECT MIN(created_at) FROM '{merged_file}'").fetchone()[0]
start_time = max(min_created_at_materials, min_created_at_merged)
end_time = start_time + datetime.timedelta(days=TIME_WINDOW_DAYS)

print(f"📅 Time range filter: {start_time} -> {end_time}")

# ✅ WRITE RESULT DIRECTLY TO PARQUET (no .df())
query = f"""
COPY (
    SELECT
        merged.measure_step_number,
        merged.measure_value,
        merged.book_state_meas,
        merged.part_number,
        merged.lower_limit,
        merged.upper_limit,
        merged.measurement_name_encoded,
        merged.measurement_unit_encoded,
        merged.is_within_limits,
        merged.book_state_book,
        merged.workstep_number_mes,
        merged.book_stamp,
        merged.part_group,
        merged.line_id,
        merged.serial_number_id,
        merged.station_id,
        materials.component_position,
        materials.component_id,
        materials.panel_position,
        materials.supplier_id,
        COALESCE(mounting_place, 'unknown') AS mounting_place,
        materials.container_number_freq
    FROM '{merged_file}' merged
    INNER JOIN '{filtered_materials_file}' materials
    ON merged.serial_number_id = materials.serial_number_id
    WHERE merged.created_at BETWEEN '{start_time}' AND '{end_time}'
      AND materials.created_at BETWEEN '{start_time}' AND '{end_time}'
) TO '{merged_full_file}' (FORMAT PARQUET);
"""

print("🚀 Executing join + direct parquet write in DuckDB...")
con.execute(query)
print(f"✅ Saved FULL merged dataset to {merged_full_file}")

con.close()

📅 Time range filter: 2025-03-01 02:21:38.306000+01:00 -> 2025-03-15 02:21:38.306000+01:00
🚀 Executing join + direct parquet write in DuckDB...
✅ Saved FULL merged dataset to M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged\merged_full.parquet


## Load and inspect merged dataset

In [4]:
con = duckdb.connect()

# View first few rows
print(con.execute(f"SELECT * FROM '{merged_full_file}' LIMIT 5").fetchdf())

# Count number of rows
row_count = con.execute(f"SELECT COUNT(*) FROM '{merged_full_file}'").fetchone()[0]
print(f"📊 Row count: {row_count}")

# Show column names and types
print(con.execute(f"DESCRIBE SELECT * FROM '{merged_full_file}'").fetchdf())

# Quick summary stats
print(con.execute(f"""
    SELECT
        COUNT(*) AS n_rows,
        MIN(book_stamp) AS min_time,
        MAX(book_stamp) AS max_time
    FROM '{merged_full_file}'
""").fetchdf())
con.close()

   measure_step_number  measure_value  book_state_meas part_number  \
0                   32           34.0                0    9d9d8f8a   
1                   32           34.0                0    9d9d8f8a   
2                   32           34.0                0    9d9d8f8a   
3                   32           34.0                0    9d9d8f8a   
4                   32           35.0                0    9d9d8f8a   

   lower_limit  upper_limit  measurement_name_encoded  \
0         33.0         37.0                     81352   
1         33.0         37.0                     81352   
2         33.0         37.0                     81352   
3         33.0         37.0                     81352   
4         33.0         37.0                     81352   

   measurement_unit_encoded  is_within_limits  book_state_book  ...  \
0                   1871126                 1                0  ...   
1                   1871126                 1                0  ...   
2                   187

## NULL VALUES CHECK

In [5]:
con = duckdb.connect()

# Get all columns
columns = [col[0] for col in con.execute(f"DESCRIBE SELECT * FROM '{merged_full_file}'").fetchall()]
print("Columns in parquet:")
print(columns)

# Check null counts per column
print("\nNull values per column:")
for col in columns:
    query = f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT({col}) AS non_nulls,
        COUNT(*) - COUNT({col}) AS nulls
    FROM '{merged_full_file}'
    """
    result = con.execute(query).fetchdf()
    print(f"{col}: nulls = {result['nulls'][0]} / {result['total_rows'][0]} total")

con.close()

Columns in parquet:
['measure_step_number', 'measure_value', 'book_state_meas', 'part_number', 'lower_limit', 'upper_limit', 'measurement_name_encoded', 'measurement_unit_encoded', 'is_within_limits', 'book_state_book', 'workstep_number_mes', 'book_stamp', 'part_group', 'line_id', 'serial_number_id', 'station_id', 'component_position', 'component_id', 'panel_position', 'supplier_id', 'mounting_place', 'container_number_freq']

Null values per column:
measure_step_number: nulls = 0 / 269973717 total
measure_value: nulls = 0 / 269973717 total
book_state_meas: nulls = 0 / 269973717 total
part_number: nulls = 0 / 269973717 total
lower_limit: nulls = 0 / 269973717 total
upper_limit: nulls = 0 / 269973717 total
measurement_name_encoded: nulls = 0 / 269973717 total
measurement_unit_encoded: nulls = 0 / 269973717 total
is_within_limits: nulls = 0 / 269973717 total
book_state_book: nulls = 0 / 269973717 total
workstep_number_mes: nulls = 0 / 269973717 total
book_stamp: nulls = 0 / 269973717 tot